# traineo1 — Conservative Kaggle Training for gardenbunga
Notebook ini sudah disesuaikan untuk repo `gardenbunga` yang sedang kita pakai sekarang.
Versi ini **tidak clone repo lain** dan **tidak download dataset**. Dataset diasumsikan sudah tersedia di disk atau mount Kaggle, jadi kita cukup isi path direktori lokal.

**Urutan cell:**
1. Config & helpers
2. Verifikasi dataset lokal
3. Verifikasi repo saat ini
4. Setup venv + install dependencies
5. Validasi GPU
6. Siapkan checkpoint pretrained / resume
7. Siapkan vocab
8. Merge CSV metadata
9. Prepare dataset
10. Mamba probe + repair
11. Install flash-attn (opsional)
12. Login W&B (opsional)
13. Verifikasi training entrypoint
14. Conservative finetune
15. Optional continue finetune
16. Push checkpoint ke Hugging Face (opsional)
17. Inference test


In [ ]:
# ── Cell 1: Config & Helpers ──────────────────────────────────────────────────
import os
import shutil
import subprocess
from pathlib import Path


def optional_str(value: str) -> str:
    return (value or "").strip()


def detect_repo_dir() -> Path:
    cwd = Path.cwd().resolve()
    for cand in [cwd, *cwd.parents]:
        if (cand / "src" / "f5_tts").exists() and (cand / "pyproject.toml").exists():
            return cand
    raise FileNotFoundError(
        "Repo gardenbunga tidak terdeteksi. Jalankan notebook ini dari dalam repo ini "
        "atau ubah fungsi detect_repo_dir()."
    )


WANDB_API_KEY_RAW = ""
HF_TOKEN_RAW = ""

WANDB_API_KEY = optional_str(WANDB_API_KEY_RAW)
HF_TOKEN = optional_str(HF_TOKEN_RAW)
ENABLE_WANDB = bool(WANDB_API_KEY)

WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "gardenbunga-conservative"

DATASET_NAME = "datasetku"
DATASET_ROOT_RAW = "/kaggle/input/tts-indo"
DATASET_SUBDIR = "data"
CSV_1_RAW = ""
CSV_2_RAW = ""

PRETRAIN_LOCAL_CKPT_RAW = ""
HF_PRETRAIN_REPO_ID = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
HF_PRETRAIN_FILENAME = "f5_tts_indo_v2.pt"

RESUME_HF_REPO_ID = ""
RESUME_HF_MODEL_LAST_FILENAME = "checkpoints/model_last.pt"
RESUME_LOCAL_CKPT_RAW = ""

RECREATE_VENV = False
FORCE_PREPARE = False
INSTALL_FLASH_ATTN = False
INFER_USE_EMA = False

ACCELERATE_NUM_PROCESSES = 1
ACCELERATE_MIXED_PRECISION = "fp16"
TRAIN_BATCH_SIZE_PER_GPU = 8000
TRAIN_MAX_SAMPLES = 64
TRAIN_NUM_WORKERS = 4
TRAIN_EPOCHS = 10
TRAIN_LR = 1e-5
TRAIN_GRAD_ACCUMULATION_STEPS = 2
TRAIN_WARMUP_UPDATES = 500

CONTINUE_EPOCHS = 20
CONTINUE_LR = 5e-6
CONTINUE_WARMUP_UPDATES = 200

TRAIN_CONFIG_NAME = "F5TTS_v1_Base_Mamba_Conservative.yaml"
TRAIN_RUN_NAME = "F5TTS_v1_Base_Mamba_Conservative_traineo1"
CONTINUE_RUN_NAME = "F5TTS_v1_Base_Mamba_Conservative_traineo1_continue"

REPO_DIR = detect_repo_dir()
WORKDIR = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else REPO_DIR
INFER_OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else (REPO_DIR / "outputs")
VENV_DIR = REPO_DIR / ".venv"
VENV_PY = VENV_DIR / "bin/python"

DATASET_ROOT = Path(DATASET_ROOT_RAW).expanduser()
DATASET_DATA_DIR = DATASET_ROOT / DATASET_SUBDIR if DATASET_SUBDIR else DATASET_ROOT
CSV_1 = Path(CSV_1_RAW).expanduser() if optional_str(CSV_1_RAW) else (DATASET_DATA_DIR / "metadata.csv")
CSV_2 = Path(CSV_2_RAW).expanduser() if optional_str(CSV_2_RAW) else None

PRETRAIN_LOCAL_CKPT = Path(PRETRAIN_LOCAL_CKPT_RAW).expanduser() if optional_str(PRETRAIN_LOCAL_CKPT_RAW) else None
RESUME_LOCAL_CKPT = Path(RESUME_LOCAL_CKPT_RAW).expanduser() if optional_str(RESUME_LOCAL_CKPT_RAW) else None

TRAIN_SAVE_DIR_REL = f"ckpts/F5TTS_v1_Base_Mamba_Conservative_vocos_pinyin_{DATASET_NAME}"
TRAIN_SAVE_DIR = REPO_DIR / TRAIN_SAVE_DIR_REL
TRAIN_LAST = TRAIN_SAVE_DIR / "model_last.pt"

if PRETRAIN_LOCAL_CKPT is not None:
    PRETRAIN_TARGET_CKPT = TRAIN_SAVE_DIR / f"pretrained_{PRETRAIN_LOCAL_CKPT.name}"
else:
    PRETRAIN_TARGET_CKPT = TRAIN_SAVE_DIR / f"pretrained_{HF_PRETRAIN_FILENAME}"

MERGED_CSV = REPO_DIR / "data" / f"{DATASET_NAME}_merged.csv"
PREPARED_DATASET_DIR = REPO_DIR / "data" / f"{DATASET_NAME}_pinyin"
EMILIA_VOCAB_DIR = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin"
EMILIA_VOCAB_PATH = EMILIA_VOCAB_DIR / "vocab.txt"
NEXT_RESUME_FILE = WORKDIR / "next_resume_links.txt"

WORKDIR.mkdir(parents=True, exist_ok=True)
INFER_OUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_SAVE_DIR.mkdir(parents=True, exist_ok=True)


def run_cmd(cmd, cwd=None, env=None, timeout=None):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print("\n$", printable)
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        timeout=timeout,
    )


def run_py(args, cwd=None, env=None, timeout=None):
    return run_cmd(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args],
        cwd=cwd,
        env=env,
        timeout=timeout,
    )


print("Config siap.")
print("REPO_DIR     :", REPO_DIR)
print("WORKDIR      :", WORKDIR)
print("INFER_OUT_DIR:", INFER_OUT_DIR)
print("DATASET_ROOT :", DATASET_ROOT)
print("TRAIN_SAVE   :", TRAIN_SAVE_DIR)


In [ ]:
# ── Cell 2: Verifikasi Dataset Lokal ───────────────────────────────────────────
print(f"Mencari dataset di {DATASET_ROOT} ...")

if not DATASET_ROOT.exists():
    possible_inputs = list(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []
    if possible_inputs:
        print(f"Isi /kaggle/input: {possible_inputs}")
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di {DATASET_ROOT}. Isi DATASET_ROOT_RAW dengan direktori yang benar."
    )

if not CSV_1.exists():
    fallback = next(iter(sorted(DATASET_ROOT.glob("**/metadata.csv"))), None)
    if fallback is None:
        raise FileNotFoundError(
            f"metadata.csv tidak ditemukan di {DATASET_ROOT}. Isi CSV_1_RAW jika nama file metadata berbeda."
        )
    CSV_1 = fallback

if CSV_2 is not None and not CSV_2.exists():
    print(f"CSV_2 tidak ditemukan, skip: {CSV_2}")
    CSV_2 = None

print("CSV_1 :", CSV_1)
print("CSV_2 :", CSV_2)
run_cmd(["ls", "-lah", str(DATASET_ROOT)])


In [ ]:
# ── Cell 3: Verifikasi Repo Saat Ini ───────────────────────────────────────────
        required = [
            REPO_DIR / "src/f5_tts/train/train.py",
            REPO_DIR / "src/f5_tts/train/datasets/prepare_csv_wavs.py",
            REPO_DIR / "src/f5_tts/configs" / TRAIN_CONFIG_NAME,
            REPO_DIR / "src/f5_tts/model/hybrid_mamba.py",
        ]
        missing = [str(p) for p in required if not p.exists()]
        if missing:
            raise FileNotFoundError("File penting repo tidak ditemukan:
" + "
".join(missing))

        print("Pakai repo lokal tanpa clone ulang:", REPO_DIR)
        run_cmd(["ls", "-lah", str(REPO_DIR)])


In [ ]:
# ── Cell 4: Setup venv + Install Dependencies ─────────────────────────────────
if shutil.which("uv") is None:
    run_cmd(["python3", "-m", "pip", "install", "-U", "uv"])

TARGET_PY_MM = "3.11"
TARGET_PY = Path(f"/usr/bin/python{TARGET_PY_MM}")

if shutil.which("apt-get") is not None:
    run_cmd(["apt-get", "update", "-y"])
    run_cmd([
        "apt-get", "install", "-y",
        f"python{TARGET_PY_MM}",
        f"python{TARGET_PY_MM}-venv",
        f"python{TARGET_PY_MM}-dev",
        "build-essential",
    ])

if RECREATE_VENV and VENV_DIR.exists():
    shutil.rmtree(VENV_DIR)

if not VENV_PY.exists():
    py_for_venv = str(TARGET_PY) if TARGET_PY.exists() else shutil.which("python3")
    if not py_for_venv:
        raise FileNotFoundError("python3 tidak ditemukan untuk membuat venv.")
    run_cmd(["uv", "venv", "--python", py_for_venv, str(VENV_DIR)])
else:
    print("Reuse existing venv:", VENV_DIR)

run_cmd([
    "uv", "pip", "install", "--python", str(VENV_PY),
    "--upgrade", "pip", "wheel", "setuptools<82",
])

torch_check = subprocess.run(
    [
        str(VENV_PY), "-c",
        "import importlib.util; "
        "spec = importlib.util.find_spec('torch'); "
        "print('missing' if spec is None else __import__('torch').__version__)"
    ],
    capture_output=True,
    text=True,
    check=True,
)
installed_torch = torch_check.stdout.strip().splitlines()[-1]
if installed_torch != "2.8.0+cu128":
    run_cmd([
        "uv", "pip", "install", "--python", str(VENV_PY),
        "--index-url", "https://download.pytorch.org/whl/cu128",
        "--extra-index-url", "https://pypi.org/simple",
        "--index-strategy", "unsafe-best-match",
        "--force-reinstall",
        "torch==2.8.0+cu128",
        "torchvision==0.23.0+cu128",
        "torchaudio==2.8.0+cu128",
    ])
else:
    print("Torch sudah cocok:", installed_torch)

run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-e", str(REPO_DIR)])
run_cmd([
    "uv", "pip", "install", "--python", str(VENV_PY),
    "mamba-ssm==2.3.1",
    "causal-conv1d>=1.4.0",
    "huggingface_hub",
])

run_cmd([str(VENV_PY), "-c",
    "import torch, torchaudio, setuptools; "
    "print('torch =', torch.__version__, 'cuda =', torch.version.cuda); "
    "print('torchaudio =', torchaudio.__version__); "
    "print('setuptools =', setuptools.__version__)"
])


In [ ]:
# ── Cell 5: Validasi GPU ──────────────────────────────────────────────────────
run_py([
    "-c",
    "import torch; "
    "print('torch', torch.__version__); "
    "print('cuda_count', torch.cuda.device_count()); "
    "[print(i, torch.cuda.get_device_name(i), 'cc', torch.cuda.get_device_capability(i)) "
    " for i in range(torch.cuda.device_count())]; "
    "assert torch.cuda.device_count() >= 1, 'GPU tidak terdeteksi'",
])

In [ ]:
# ── Cell 6: Siapkan Pretrained / Resume Checkpoint ───────────────────────────
        TRAIN_SAVE_DIR.mkdir(parents=True, exist_ok=True)

        download_env = os.environ.copy()
        if HF_TOKEN:
            download_env["HF_TOKEN"] = HF_TOKEN

        resolve_script = f"""
from pathlib import Path
import os, shutil
from huggingface_hub import hf_hub_download

train_last = Path(r"{TRAIN_LAST}")
resume_local = {str(RESUME_LOCAL_CKPT) if RESUME_LOCAL_CKPT else None!r}
resume_repo = {RESUME_HF_REPO_ID!r}
resume_file = {RESUME_HF_MODEL_LAST_FILENAME!r}
pretrain_local = {str(PRETRAIN_LOCAL_CKPT) if PRETRAIN_LOCAL_CKPT else None!r}
pretrain_target = Path(r"{PRETRAIN_TARGET_CKPT}")
hf_pretrain_repo = {HF_PRETRAIN_REPO_ID!r}
hf_pretrain_file = {HF_PRETRAIN_FILENAME!r}
token = os.environ.get("HF_TOKEN") or None

def maybe_copy(src, dst):
    src = Path(src).expanduser().resolve()
    if not src.exists():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src != dst.resolve():
        shutil.copy2(src, dst)
    print(f"copied: {{src}} -> {{dst}}")

if train_last.exists():
    print("Resume lokal sudah ada:", train_last)
elif resume_local:
    maybe_copy(resume_local, train_last)
elif resume_repo:
    src = Path(hf_hub_download(repo_id=resume_repo, filename=resume_file, token=token))
    train_last.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, train_last)
    print("Resume HF ->", train_last)
elif pretrain_local:
    maybe_copy(pretrain_local, pretrain_target)
elif pretrain_target.exists():
    print("Pretrained sudah ada:", pretrain_target)
else:
    src = Path(hf_hub_download(repo_id=hf_pretrain_repo, filename=hf_pretrain_file, token=token))
    pretrain_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, pretrain_target)
    print("Pretrain HF ->", pretrain_target)
"""

        run_py(["-c", resolve_script], cwd=REPO_DIR, env=download_env)
        run_cmd(["ls", "-lah", str(TRAIN_SAVE_DIR)])


In [ ]:
# ── Cell 7: Siapkan Vocab (Emilia_ZH_EN_pinyin/vocab.txt) ────────────────────
EMILIA_VOCAB_DIR  = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin"
EMILIA_VOCAB_PATH = EMILIA_VOCAB_DIR / "vocab.txt"
EMILIA_VOCAB_DIR.mkdir(parents=True, exist_ok=True)

if EMILIA_VOCAB_PATH.exists() and EMILIA_VOCAB_PATH.stat().st_size > 0:
    print("Vocab sudah ada:", EMILIA_VOCAB_PATH)
else:
    vocab_script = f"""
from pathlib import Path
import shutil
from huggingface_hub import hf_hub_download

target = Path(r'{EMILIA_VOCAB_PATH}')
target.parent.mkdir(parents=True, exist_ok=True)

candidates = [
    ('SWivid/F5-TTS', 'F5TTS_Base/vocab.txt'),
    ('SWivid/F5-TTS', 'F5TTS_v1_Base/vocab.txt'),
]
for repo_id, filename in candidates:
    try:
        src = Path(hf_hub_download(repo_id=repo_id, filename=filename))
        shutil.copy2(src, target)
        print(f'Vocab dari {{repo_id}}/{{filename}} -> {{target}}')
        break
    except Exception as e:
        print(f'Gagal dari {{repo_id}}/{{filename}}: {{e}}')
else:
    raise RuntimeError('Gagal download vocab.')

print('vocab size:', target.stat().st_size)
"""
    run_py(["-c", vocab_script], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(EMILIA_VOCAB_DIR)])

In [ ]:
# ── Cell 8: Merge CSV Metadata ────────────────────────────────────────────────
# prepare_csv_wavs.py mensyaratkan:
#   1. Baris header: audio_file|text
#   2. audio_file berupa absolute path
import csv
import pandas as pd


def _load_metadata(csv_path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(csv_path, sep="|", dtype=str)
        if {"audio_file", "text"}.issubset(df.columns):
            return df[["audio_file", "text"]]
    except Exception:
        pass
    return pd.read_csv(csv_path, sep="|", header=None, names=["audio_file", "text"], dtype=str)


def _resolve_audio_path(p: str, csv_source: Path) -> str:
    p = str(p).strip().replace("\", "/")
    candidate = Path(p)

    if candidate.is_absolute() and candidate.exists():
        return str(candidate)

    rel_to_csv = csv_source.parent / p
    if rel_to_csv.exists():
        return str(rel_to_csv.resolve())

    rel_to_data = DATASET_DATA_DIR / p
    if rel_to_data.exists():
        return str(rel_to_data.resolve())

    rel_to_root = DATASET_ROOT / p
    if rel_to_root.exists():
        return str(rel_to_root.resolve())

    return str((DATASET_DATA_DIR / p).resolve())


dfs = []
for csv_path in filter(None, [CSV_1, CSV_2]):
    if not csv_path.exists():
        print(f"Skip (tidak ada): {csv_path}")
        continue
    df = _load_metadata(csv_path)
    df = df.dropna(subset=["audio_file", "text"])
    df["audio_file"] = df["audio_file"].astype(str).apply(lambda p: _resolve_audio_path(p, csv_path))
    df["text"] = df["text"].astype(str)
    df = df[df["text"].str.strip() != ""]
    dfs.append(df)
    print(f"Loaded {len(df)} rows dari {csv_path.name}")

    non_abs = df[~df["audio_file"].str.startswith("/")]
    if not non_abs.empty:
        print(f"  WARNING: {len(non_abs)} path masih non-absolute, contoh: {non_abs.iloc[0]['audio_file']}")

if not dfs:
    raise RuntimeError("Tidak ada CSV yang berhasil di-load.")

merged = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=["audio_file"])
MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(MERGED_CSV, sep="|", index=False, header=True, quoting=csv.QUOTE_NONE, escapechar="\")

sample = merged["audio_file"].iloc[0]
print(f"\nMerged rows : {len(merged)}")
print(f"Sample path : {sample}")
print(f"Is absolute : {Path(sample).is_absolute()}")
print(f"Saved       : {MERGED_CSV}")

with open(MERGED_CSV, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 4:
            break
        print(f"  [{i}] {line.rstrip()}")


In [ ]:
# ── Cell 9: Prepare Dataset ───────────────────────────────────────────────────
PREPARED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

try:
    run_py(["-c", "import f5_tts; print('f5_tts ok')"], cwd=REPO_DIR)
except subprocess.CalledProcessError:
    print("Install f5_tts editable...")
    run_cmd([
        "uv", "pip", "install", "--python", str(VENV_PY), "-e", str(REPO_DIR),
    ], cwd=REPO_DIR)

prepared_ok = (PREPARED_DATASET_DIR / "raw.arrow").exists() and (PREPARED_DATASET_DIR / "duration.json").exists()
if prepared_ok and not FORCE_PREPARE:
    print("Prepared dataset sudah ada, skip:", PREPARED_DATASET_DIR)
else:
    run_py([
        "src/f5_tts/train/datasets/prepare_csv_wavs.py",
        str(MERGED_CSV),
        str(PREPARED_DATASET_DIR),
        "--workers", str(TRAIN_NUM_WORKERS),
    ], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(PREPARED_DATASET_DIR)])


In [ ]:
# ── Cell 10: Mamba Probe + Repair ─────────────────────────────────────────────
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"]  = WANDB_ENTITY
env["WANDB_PROJECT"] = WANDB_PROJECT

# setup LD_LIBRARY_PATH dari venv site-packages
site_pkgs = sorted((VENV_DIR / "lib").glob("python*/site-packages"))
if site_pkgs:
    sp = site_pkgs[-1]
    cuda_rel = [
        "nvidia/cublas/lib", "nvidia/cuda_runtime/lib", "nvidia/cudnn/lib",
        "nvidia/cufft/lib",  "nvidia/nccl/lib",         "nvidia/nvjitlink/lib",
    ]
    cuda_libs = [str(sp / r) for r in cuda_rel if (sp / r).exists()]
    if cuda_libs:
        env["LD_LIBRARY_PATH"] = ":".join(cuda_libs + [env.get("LD_LIBRARY_PATH", "")])

env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["PYTHONFAULTHANDLER"]       = "1"
runtime_env = env.copy()


def run_py_nosync(args, cwd=None, timeout=None):
    return run_cmd(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args],
        cwd=cwd, env=runtime_env, timeout=timeout,
    )


MAMBA_PROBE_TIMEOUT = 90

def _probe_mamba(timeout=MAMBA_PROBE_TIMEOUT):
    run_py_nosync(["-u", "-c",
        "import torch; import mamba_ssm; import selective_scan_cuda; "
        "print('mamba probe ok')"
    ], cwd=REPO_DIR, timeout=timeout)


def _repair_mamba():
    print("Repair mamba...")

    # pastikan python headers
    py_mm = subprocess.check_output(
        [str(VENV_PY), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
        text=True,
    ).strip()
    py_h = Path(f"/usr/include/python{py_mm}/Python.h")
    if not py_h.exists():
        run_cmd(["apt-get", "update", "-y"])
        run_cmd(["apt-get", "install", "-y", f"python{py_mm}-dev", "build-essential"])

    run_cmd([str(VENV_PY), "-m", "pip", "install",
             "--upgrade", "pip", "wheel", "ninja", "setuptools<82"],
             cwd=REPO_DIR, env=runtime_env)

    # coba wheel binary dulu
    wheel_env = runtime_env.copy()
    arch_probe = subprocess.run(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", "-c",
         "import torch; c=torch.cuda.get_device_capability(0); print(f'{c[0]}.{c[1]}')"],
        cwd=str(REPO_DIR), env=runtime_env, capture_output=True, text=True,
    )
    cuda_arch = arch_probe.stdout.strip().splitlines()[-1] if arch_probe.returncode == 0 else "7.5"
    wheel_env["TORCH_CUDA_ARCH_LIST"] = cuda_arch
    wheel_env["MAX_JOBS"] = "4"

    try:
        run_cmd([
            "uv", "pip", "install", "--python", str(VENV_PY),
            "--force-reinstall", "--no-cache-dir", "--prefer-binary",
            "--no-build-isolation", "--no-deps",
            "causal-conv1d", "mamba-ssm",
        ], cwd=REPO_DIR, env=wheel_env)
        _probe_mamba(timeout=120)
        print("Mamba OK (wheel).")
        return
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Wheel gagal, build from source...")

    build_env = wheel_env | {"MAMBA_FORCE_BUILD": "TRUE", "CAUSAL_CONV1D_FORCE_BUILD": "TRUE"}
    for pkg in ["causal-conv1d", "mamba-ssm"]:
        run_cmd([
            str(VENV_PY), "-m", "pip", "install",
            "--no-cache-dir", "--no-build-isolation",
            "--force-reinstall", "--no-binary", ":all:", "--no-deps", pkg,
        ], cwd=REPO_DIR, env=build_env)

    _probe_mamba(timeout=240)
    print("Mamba OK (built from source).")


try:
    _probe_mamba()
    print("mamba_mode: enabled (fast-path)")
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print("Probe gagal — repair...")
    _repair_mamba()
    _probe_mamba(timeout=240)
    print("mamba_mode: enabled (after repair)")

print("Runtime siap.")

In [ ]:
# ── Cell 11: Install Flash-Attention (Opsional) ───────────────────────────────
if not INSTALL_FLASH_ATTN:
    USE_FLASH_ATTN = False
    print("INSTALL_FLASH_ATTN = False -> skip, pakai backend torch.")
else:
    FLASH_WHL = (
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
        "flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl"
    )

    install_env = runtime_env.copy()
    install_env["PIP_NO_DEPS"] = "1"

    try:
        run_cmd(
            ["uv", "run", "--no-sync", "--python", str(VENV_PY),
             "pip", "uninstall", "-y", "flash-attn", "flash_attn"],
            cwd=REPO_DIR, env=install_env,
        )
        run_cmd(
            ["uv", "run", "--no-sync", "--python", str(VENV_PY),
             "pip", "install", "--no-deps", "--force-reinstall", FLASH_WHL],
            cwd=REPO_DIR, env=install_env,
        )
        run_py_nosync(
            ["-c", "import flash_attn, torch; "
             "print('flash_attn', flash_attn.__version__, 'torch', torch.__version__)"],
            cwd=REPO_DIR,
        )
        USE_FLASH_ATTN = True
    except subprocess.CalledProcessError:
        USE_FLASH_ATTN = False
        print("flash_attn gagal di-install, fallback ke torch.")

print("USE_FLASH_ATTN =", USE_FLASH_ATTN)


In [ ]:
# ── Cell 12: Login W&B (Opsional) ─────────────────────────────────────────────
        if not ENABLE_WANDB:
            print("WANDB disabled. Isi WANDB_API_KEY_RAW jika ingin logging ke W&B.")
        else:
            wandb_env = runtime_env.copy()
            wandb_env["WANDB_API_KEY"] = WANDB_API_KEY
            wandb_env["WANDB_ENTITY"] = WANDB_ENTITY
            wandb_env["WANDB_PROJECT"] = WANDB_PROJECT

            wandb_smoke = """
import os, wandb
wandb.login(key=os.environ["WANDB_API_KEY"])
run = wandb.init(
    entity=os.environ["WANDB_ENTITY"],
    project=os.environ["WANDB_PROJECT"],
    config={"smoke": True},
)
run.log({"smoke_loss": 0.0})
run.finish()
print("wandb sanity done")
""".strip()

            run_py(["-c", wandb_smoke], cwd=REPO_DIR, env=wandb_env)


In [ ]:
# ── Cell 13: Verifikasi Training Entrypoint ───────────────────────────────────
        required = [
            REPO_DIR / "src/f5_tts/train/train.py",
            REPO_DIR / "src/f5_tts/configs" / TRAIN_CONFIG_NAME,
            MERGED_CSV,
            PREPARED_DATASET_DIR / "duration.json",
            PREPARED_DATASET_DIR / "raw.arrow",
            PREPARED_DATASET_DIR / "vocab.txt",
        ]
        missing = [str(p) for p in required if not p.exists()]
        if missing:
            raise FileNotFoundError("Asset training belum siap:
" + "
".join(missing))

        print("train.py :", REPO_DIR / "src/f5_tts/train/train.py")
        print("config   :", REPO_DIR / "src/f5_tts/configs" / TRAIN_CONFIG_NAME)
        print("dataset  :", PREPARED_DATASET_DIR)
        print("resume   :", TRAIN_LAST if TRAIN_LAST.exists() else "(none)")
        print("pretrain :", PRETRAIN_TARGET_CKPT if PRETRAIN_TARGET_CKPT.exists() else "(none)")


In [ ]:
# ── Cell 14: Conservative Finetune ────────────────────────────────────────────
NUM_WORKERS = TRAIN_NUM_WORKERS

train_env = runtime_env.copy()
train_env.update({
    "OMP_NUM_THREADS": str(NUM_WORKERS),
    "MKL_NUM_THREADS": str(NUM_WORKERS),
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,roundup_power2_divisions:16",
    "TORCH_ALLOW_TF32_CUBLAS_OVERRIDE": "1",
    "PYTHONFAULTHANDLER": "1",
})
if ENABLE_WANDB:
    train_env["WANDB_API_KEY"] = WANDB_API_KEY
    train_env["WANDB_ENTITY"] = WANDB_ENTITY
    train_env["WANDB_PROJECT"] = WANDB_PROJECT

attn_override = ["model.arch.attn_backend=flash_attn"] if USE_FLASH_ATTN else ["model.arch.attn_backend=torch"]
logger_override = (
    [
        "ckpts.logger=wandb",
        f"ckpts.wandb_project={WANDB_PROJECT}",
        f"ckpts.wandb_run_name={TRAIN_RUN_NAME}",
    ]
    if ENABLE_WANDB
    else ["ckpts.logger=null"]
)

train_cmd = [
    "uv", "run", "--no-sync", "--python", str(VENV_PY),
    "accelerate", "launch",
    f"--num_processes={ACCELERATE_NUM_PROCESSES}",
    f"--mixed_precision={ACCELERATE_MIXED_PRECISION}",
    "--dynamo_backend=no",
    "src/f5_tts/train/train.py",
    "--config-name", TRAIN_CONFIG_NAME,
    f"datasets.name={DATASET_NAME}",
    f"datasets.batch_size_per_gpu={TRAIN_BATCH_SIZE_PER_GPU}",
    f"datasets.max_samples={TRAIN_MAX_SAMPLES}",
    f"datasets.num_workers={NUM_WORKERS}",
    f"optim.epochs={TRAIN_EPOCHS}",
    f"optim.learning_rate={TRAIN_LR}",
    f"optim.num_warmup_updates={TRAIN_WARMUP_UPDATES}",
    f"optim.grad_accumulation_steps={TRAIN_GRAD_ACCUMULATION_STEPS}",
    "model.arch.checkpoint_activations=True",
    f"ckpts.save_dir={TRAIN_SAVE_DIR_REL}",
    "ckpts.save_per_updates=5000",
    "ckpts.last_per_updates=500",
    "ckpts.keep_last_n_checkpoints=2",
    "ckpts.log_samples=False",
    *logger_override,
    *attn_override,
]

if TRAIN_LAST.exists():
    print("model_last.pt sudah ada -> train.py akan auto-resume dari sana.")
elif PRETRAIN_TARGET_CKPT.exists():
    print("Pretrained tersedia -> train.py akan load dari", PRETRAIN_TARGET_CKPT)
else:
    print("Tidak ada pretrained / resume checkpoint -> training mulai dari nol.")

run_cmd(train_cmd, cwd=REPO_DIR, env=train_env)
run_cmd(["ls", "-lah", str(TRAIN_SAVE_DIR)])

if not TRAIN_LAST.exists():
    raise FileNotFoundError(f"Checkpoint training tidak ditemukan: {TRAIN_LAST}")

print("Training selesai:", TRAIN_LAST)


In [ ]:
# ── Cell 15: Optional Continue Finetune ───────────────────────────────────────
if not TRAIN_LAST.exists():
    raise FileNotFoundError(f"Resume checkpoint tidak ada: {TRAIN_LAST}")

continue_env = runtime_env.copy()
continue_env.update({
    "OMP_NUM_THREADS": str(TRAIN_NUM_WORKERS),
    "MKL_NUM_THREADS": str(TRAIN_NUM_WORKERS),
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,roundup_power2_divisions:16",
    "TORCH_ALLOW_TF32_CUBLAS_OVERRIDE": "1",
    "PYTHONFAULTHANDLER": "1",
})
if ENABLE_WANDB:
    continue_env["WANDB_API_KEY"] = WANDB_API_KEY
    continue_env["WANDB_ENTITY"] = WANDB_ENTITY
    continue_env["WANDB_PROJECT"] = WANDB_PROJECT

attn_override = ["model.arch.attn_backend=flash_attn"] if USE_FLASH_ATTN else ["model.arch.attn_backend=torch"]
logger_override = (
    [
        "ckpts.logger=wandb",
        f"ckpts.wandb_project={WANDB_PROJECT}",
        f"ckpts.wandb_run_name={CONTINUE_RUN_NAME}",
    ]
    if ENABLE_WANDB
    else ["ckpts.logger=null"]
)

continue_cmd = [
    "uv", "run", "--no-sync", "--python", str(VENV_PY),
    "accelerate", "launch",
    f"--num_processes={ACCELERATE_NUM_PROCESSES}",
    f"--mixed_precision={ACCELERATE_MIXED_PRECISION}",
    "--dynamo_backend=no",
    "src/f5_tts/train/train.py",
    "--config-name", TRAIN_CONFIG_NAME,
    f"datasets.name={DATASET_NAME}",
    f"datasets.batch_size_per_gpu={TRAIN_BATCH_SIZE_PER_GPU}",
    f"datasets.max_samples={TRAIN_MAX_SAMPLES}",
    f"datasets.num_workers={TRAIN_NUM_WORKERS}",
    f"optim.epochs={CONTINUE_EPOCHS}",
    f"optim.learning_rate={CONTINUE_LR}",
    f"optim.num_warmup_updates={CONTINUE_WARMUP_UPDATES}",
    f"optim.grad_accumulation_steps={TRAIN_GRAD_ACCUMULATION_STEPS}",
    "model.arch.checkpoint_activations=True",
    f"ckpts.save_dir={TRAIN_SAVE_DIR_REL}",
    "ckpts.save_per_updates=5000",
    "ckpts.last_per_updates=500",
    "ckpts.keep_last_n_checkpoints=2",
    "ckpts.log_samples=False",
    *logger_override,
    *attn_override,
]

run_cmd(continue_cmd, cwd=REPO_DIR, env=continue_env)
run_cmd(["ls", "-lah", str(TRAIN_SAVE_DIR)])

if not TRAIN_LAST.exists():
    raise FileNotFoundError(f"Checkpoint continue training tidak ditemukan: {TRAIN_LAST}")

print("Continue training selesai:", TRAIN_LAST)


In [ ]:
# ── Cell 16: Push Checkpoint ke Hugging Face (Opsional) ───────────────────────
from datetime import datetime, timezone
from uuid import uuid4

if not HF_TOKEN:
    raise ValueError("HF_TOKEN_RAW masih kosong. Isi token kalau ingin upload checkpoint.")
if not TRAIN_LAST.exists():
    raise FileNotFoundError(f"Checkpoint belum ada: {TRAIN_LAST}")

try:
    from huggingface_hub import HfApi
except ImportError:
    run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])
    from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
owner = api.whoami(token=HF_TOKEN).get("name", "").strip()
if not owner:
    raise RuntimeError("Owner HF tidak terdeteksi dari token.")

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
target_repo_id = None
for _ in range(5):
    candidate = f"{owner}/gardenbunga-conservative-{timestamp}-{uuid4().hex[:8]}"
    try:
        api.create_repo(repo_id=candidate, repo_type="model", private=True, exist_ok=False, token=HF_TOKEN)
        target_repo_id = candidate
        break
    except Exception as exc:
        if not any(t in str(exc).lower() for t in ("already exists", "409", "conflict")):
            raise

if target_repo_id is None:
    raise RuntimeError("Gagal membuat repo HF baru.")

print("Target repo:", target_repo_id)

api.upload_file(
    path_or_fileobj=str(TRAIN_LAST),
    path_in_repo="checkpoints/model_last.pt",
    repo_id=target_repo_id,
    repo_type="model",
    token=HF_TOKEN,
)
api.upload_file(
    path_or_fileobj=str(REPO_DIR / "src/f5_tts/configs" / TRAIN_CONFIG_NAME),
    path_in_repo=f"configs/{TRAIN_CONFIG_NAME}",
    repo_id=target_repo_id,
    repo_type="model",
    token=HF_TOKEN,
)
api.upload_file(
    path_or_fileobj=str(PREPARED_DATASET_DIR / "vocab.txt"),
    path_in_repo="configs/vocab.txt",
    repo_id=target_repo_id,
    repo_type="model",
    token=HF_TOKEN,
)

repo_url = f"https://huggingface.co/{target_repo_id}"
model_last_url = f"{repo_url}/blob/main/checkpoints/model_last.pt"

NEXT_RESUME_FILE.write_text(
    "\n".join([
        "Hardcode untuk run berikutnya:",
        f'RESUME_HF_REPO_ID = "{target_repo_id}"',
        'RESUME_HF_MODEL_LAST_FILENAME = "checkpoints/model_last.pt"',
        "",
        f"Repo URL       : {repo_url}",
        f"model_last URL : {model_last_url}",
    ]),
    encoding="utf-8",
)

print("Upload selesai.")
print("Repo URL :", repo_url)
print("Next resume links:", NEXT_RESUME_FILE)


In [ ]:
# ── Cell 17: Inference Test ───────────────────────────────────────────────────
        import csv as _csv
        from IPython.display import Audio, display

        ckpt_file = TRAIN_LAST if TRAIN_LAST.exists() else PRETRAIN_TARGET_CKPT
        if not ckpt_file.exists():
            raise FileNotFoundError(f"Checkpoint tidak ada: {ckpt_file}")

        with open(MERGED_CSV, "r", encoding="utf-8") as f:
            rows = [
                r for r in _csv.DictReader(f, delimiter="|")
                if r.get("audio_file", "").strip() and r.get("text", "").strip()
            ]

        if not rows:
            raise ValueError(f"metadata kosong: {MERGED_CSV}")

        ref_row = rows[0]
        ref_text = ref_row["text"].strip()
        gen_text = "halo, selamat datang di gardenbunga"

        def _find_audio(p: str) -> Path:
            for base in [Path(p), DATASET_ROOT / p, DATASET_DATA_DIR / p, REPO_DIR / p, WORKDIR / p]:
                if Path(base).exists():
                    return Path(base).resolve()
            raise FileNotFoundError(f"Audio referensi tidak ketemu: {p}")

        ref_audio = _find_audio(ref_row["audio_file"])
        vocab_file = PREPARED_DATASET_DIR / "vocab.txt"
        if not vocab_file.exists():
            raise FileNotFoundError(f"vocab.txt tidak ditemukan: {vocab_file}")

        out_wav = INFER_OUT_DIR / "infer_traineo1_test.wav"

        infer_script = f"""
from pathlib import Path
import soundfile as sf
from omegaconf import OmegaConf
from f5_tts.infer.utils_infer import load_model, load_vocoder, preprocess_ref_audio_text, infer_process
from f5_tts.model.backbones.dit import DiT

repo_dir = Path(r"{REPO_DIR}")
cfg = OmegaConf.load(str(repo_dir / "src/f5_tts/configs/{TRAIN_CONFIG_NAME}"))
model_cfg = OmegaConf.to_container(cfg.model.arch, resolve=True)

ckpt_file = Path(r"{ckpt_file}")
vocab_file = Path(r"{vocab_file}")
ref_audio = Path(r"{ref_audio}")
out_wav = Path(r"{out_wav}")
ref_text = {ref_text!r}
gen_text = {gen_text!r}

ema_model = load_model(
    DiT,
    model_cfg,
    str(ckpt_file),
    vocab_file=str(vocab_file),
    use_ema={INFER_USE_EMA!r},
)
vocoder = load_vocoder(vocoder_name=cfg.model.mel_spec.mel_spec_type, device=str(ema_model.device))
ref_audio_proc, ref_text_proc = preprocess_ref_audio_text(str(ref_audio), ref_text)
wav, sr, spec = infer_process(
    str(ref_audio_proc),
    ref_text_proc,
    gen_text,
    ema_model,
    vocoder,
    mel_spec_type=cfg.model.mel_spec.mel_spec_type,
    device=str(ema_model.device),
)
sf.write(out_wav, wav, sr)
print("DONE:", out_wav)
"""

        run_py(["-c", infer_script], cwd=REPO_DIR, env={**runtime_env, "MPLBACKEND": "Agg"})

        print("Saved:", out_wav)
        display(Audio(str(out_wav)))


## Notes

- Notebook ini diasumsikan dijalankan dari repo `gardenbunga` yang sudah ada, jadi **tidak clone repo lain**.
- Dataset **tidak di-download**. Isi `DATASET_ROOT_RAW`, dan opsional `CSV_1_RAW` / `CSV_2_RAW` kalau nama file metadata berbeda.
- Jalur training yang dipakai sekarang adalah [train.py](/home/anekazek/anything-notsupported-inwindows/gardenbunga/src/f5_tts/train/train.py) dengan config [F5TTS_v1_Base_Mamba_Conservative.yaml](/home/anekazek/anything-notsupported-inwindows/gardenbunga/src/f5_tts/configs/F5TTS_v1_Base_Mamba_Conservative.yaml).
- Checkpoint pretrained ditempatkan sebagai `pretrained_*.pt` di folder checkpoint target agar loader repo ini bisa memakainya otomatis tanpa patch tambahan.
- `Cell 14` adalah fine-tune konservatif utama. `Cell 15` opsional kalau mau lanjut dari `model_last.pt` yang sudah ada.
- `Cell 11`, `Cell 12`, dan `Cell 16` opsional. Default awal dibuat aman: `INSTALL_FLASH_ATTN=False`, `ENABLE_WANDB=False`, `ACCELERATE_NUM_PROCESSES=1`, dan `fp16`.
- Kalau kamu memang pakai 2xT4 di Kaggle, naikkan `ACCELERATE_NUM_PROCESSES=2` secara manual sebelum training.
